<a href="https://colab.research.google.com/github/shangbingo888/AI-EDF-NOTEBOOKS-CN/blob/main/12-case-studies-and-capstone/04-deepfake-image-detection.zh.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 案例研究 D —— 面向 EU AI Act 第 50(4) 条的深度伪造检测：detector、VLM，还是两者都要？

**目标：** 在一小批精选图像上，比较三种标记**未披露的 deepfake（深度伪造）**的方式——这正是 EU AI Act 第 50(4) 条实际监管的东西。然后再把这套同样的比较指向任何你想要的图像。

本项目是 [AI-EDF-NOTEBOOKS-CN](https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN) 的一部分。

> **先把论点摆出来：**「这张图是 AI 生成的吗？」和「这张图违反了第 50(4) 条吗？」是两个不同的问题。前者是感知问题，训练过的 detector 应该能胜过通用模型。后者还需要*含义*（它是否描绘了一个真实的人或事件？是否明显是讽刺？）和*来源*（它是否已经披露过？），这两点没有任何像素分类器能回答。所以最可能的赢家是混合方案。

这篇 notebook 分六步走：**1** 诉求，**2** 数据，**3** 方法，**4** 每个工具在数据上的表现，**5** 系统对比，**6** 你自己的图像。

多模态不在核心主线上（[课程决策 #3](../docs/curriculum-plan.zh.md)）；这个案例研究把评测的思路（[02](../02-evals-basics/01-measuring-outputs.zh.ipynb)、[04](../04-evals/01-golden-sets.zh.ipynb)）和一种新模态组合起来。这些 detector 就是普通的图像分类器，所以它们用 `transformers` 加载，也正是你会实际使用的那个工具。

## 准备

在 [console.groq.com](https://console.groq.com/) 领免费 key；在 Colab 里通过**钥匙图标** → secret `GROQ_API_KEY` 添加。（完整走查：[00-setup](../00-setup/00-environment.zh.ipynb)。）一个 `HF_TOKEN` secret（免费的 Hugging Face 读 token）是可选的；它能提高 Hugging Face 的速率限制。

在 CPU runtime 上就能跑。完整跑一遍会下载两个 detector（约 0.6 GB）和 7 张示例图像，并发出约 8 次 VLM 调用（每次约 1,600 token），完全在 Groq 免费每日额度之内。

In [ ]:
%pip install -q "git+https://github.com/shangbingo888/AI-EDF-NOTEBOOKS-CN.git@main"

In [ ]:
import base64, hashlib, io, json, os, re, time, urllib.error, urllib.parse, urllib.request
from html import escape
from IPython.display import HTML, display
from PIL import Image
from aien import setup

client, TEXT_MODEL = setup()
VISION_MODEL = "qwen/qwen3.8-27b"  # Groq 的图像能力模型；如果调用 404，见 console.groq.com/docs/vision
MAX_SIDE = 512                     # 每个模型看到的图像都会降采样到这个尺寸
CACHE_PATH = "case_d_cache.json"   # VLM 的回答，这样重跑不会重复一次付费调用
SAMPLE_DIR = "case_d_samples"      # 下载下来的示例图像；见第 2 步的许可证说明

def http_get(url, headers=None, tries=5):
    """带退避的 GET。Hugging Face 会限制突发流量：遇到 429 时，服务器让等多久就等多久。"""
    headers = {"User-Agent": "ai-engineer-notebooks (case study D)", **(headers or {})}
    for attempt in range(1, tries + 1):
        try:
            with urllib.request.urlopen(urllib.request.Request(url, headers=headers), timeout=60) as r:
                return r.read()
        except urllib.error.HTTPError as e:
            if e.code not in (429, 500, 502, 503, 504) or attempt == tries:
                raise
            ra = e.headers.get("Retry-After") or ""
            wait = int(ra) if ra.isdigit() else 10 * attempt
            print(f"   HTTP {e.code}; waiting {wait}s ({attempt}/{tries})")
        except (urllib.error.URLError, TimeoutError) as e:
            if attempt == tries:
                raise
            wait = 10 * attempt
            print(f"   {type(e).__name__}; waiting {wait}s ({attempt}/{tries})")
        time.sleep(wait)

def view(raw, side=MAX_SIDE):
    """模型看到的像素：解码后、RGB、降采样。文件元数据过不了这一关。"""
    img = Image.open(io.BytesIO(raw))
    img.draft("RGB", (side, side))     # JPEG 可以直接按缩小后的尺寸解码
    img = img.convert("RGB")
    img.thumbnail((side, side))
    return img

def thumb(raw, side=110):
    """一个用于结果表格的小型内联 <img>。"""
    buf = io.BytesIO()
    view(raw, side).save(buf, format="JPEG", quality=85)
    return f'<img src="data:image/jpeg;base64,{base64.b64encode(buf.getvalue()).decode()}">'

def show_table(rows):
    """把一个 dict 列表渲染成 HTML 表格；以 <img 开头的值会显示为图像。"""
    if not rows:
        return
    cols = list(rows[0])
    cell = lambda v: str(v) if str(v).startswith("<img") else escape(str(v))
    head = "".join(f"<th>{escape(c)}</th>" for c in cols)
    body = "".join("<tr>" + "".join(f"<td>{cell(r[c])}</td>" for c in cols) + "</tr>" for r in rows)
    display(HTML(f"<table><tr>{head}</tr>{body}</table>"))

def mark(ok):
    return "✓" if ok else "✗"

CACHE = json.load(open(CACHE_PATH)) if os.path.exists(CACHE_PATH) else {}

def cached(key, fn):
    if key not in CACHE:
        CACHE[key] = fn()
        with open(CACHE_PATH, "w") as f:
            json.dump(CACHE, f)
    return CACHE[key]

## 1. 诉求

一个信任与安全团队：*「第 50 条从 2026 年 8 月 2 日起适用。找出我们平台上本应被打标、却没有的那些 deepfake。」* 在动手构建之前，先读这条法律条文（[11/01 需求界定](../11-customer-craft/01-scoping-and-discovery.zh.ipynb)）。

第 50(4) 条把义务放在**部署方**（deployer，即发布内容的人）身上：披露某个 deepfake 是生成或经过操纵的。明显属于艺术、讽刺或虚构的作品只需要有限的披露。**deepfake**（第 3(60) 条）是指生成或经过操纵的内容，它类似现有的人、物、地点、实体或事件，并且会让人误以为是真实的。所以一张图只有在**以下三点全部成立**时才是 50(4) 问题：

1. 它由 AI 生成或经 AI 操纵（*感知*）；
2. 它描绘了一个真实的、可识别的实体或事件，且并非明显虚构或讽刺（*含义*）；
3. 发布的文件**没有任何披露**：没有可见标签，没有机器可读的标记（*来源*）。

另外，第 50(2) 条要求生成器的**提供方**用机器可读的方式标记其输出，而委员会 2026 年 6 月的 Code of Practice 要求至少两层标记（例如元数据加水印）。这就是为什么条件 3 有部分可以从文件本身检查。

**范围：** 仅限静态图像（50(4) 也覆盖音频和视频）。这个系统**标记出来供人工复核**；判定某件事是否违规是一个法律判断，这里的一切都不构成法律建议。

## 2. 数据

**[OpenFake](https://huggingface.co/datasets/ComplexDataLab/OpenFake)**（`core` 子集）有来自 75 个来源的 240 万张真实和 AI 生成图像，每张都标了 `real` 或 `fake`，并附上生成器的名字。其中的真实实体 deepfake 很稀少（约每 40 张 fake 里 1 张），所以我们不从里面采样，而是从它的 test split 里**手挑**七张：

- 两张真实实体的写实 fake：一场 Real Madrid 新闻发布会，以及 *The Daily Show* 演播室；
- 一张穿「Bernie for President」T 恤的匿名男子的 AI 图像（一个需要判断的案例，见第 5 步）；
- 一张动漫插画（明显虚构）；
- 三张真实照片。

我们假设每张图像都是**在没有任何披露的情况下**发布到平台上的，所以条件 3 对全部七张都成立，问题只剩下条件 1 和 2。每张图像旁边，我们写下 **gold** 答案，也就是一位谨慎的人工复核者会做出的判断：违规 = fake **且**真实实体 **且**非虚构。`real`/`fake` 标签来自数据集；「真实实体」和「虚构」是手工标注的，这正是一个小黄金集应有的样子（[04/01](../04-evals/01-golden-sets.zh.ipynb)）。

每张图像通过 Hugging Face 的 dataset viewer API 按行号获取：每张图像两个小请求，保存到磁盘，所以重跑不会再下载任何东西。

> **⚠️ 生产现实 ——** OpenFake 是 **CC BY-NC 4.0**（非商业），并且包含真实公众人物的图像。把 `SAMPLE_DIR` 保持私有，不要重新分发，用完就删掉。

In [ ]:
DATASET_ID, SUBSET, SPLIT = "ComplexDataLab/OpenFake", "core", "test"
VIEWER = "https://datasets-server.huggingface.co"   # Hugging Face 的 dataset viewer API
HF_TOKEN = os.environ.get("HF_TOKEN")
try:
    from google.colab import userdata
    HF_TOKEN = HF_TOKEN or userdata.get("HF_TOKEN")
except Exception:                  # 不在 Colab 里，或者没有 HF_TOKEN secret：匿名也可以
    pass
HF_HEADERS = {"Authorization": f"Bearer {HF_TOKEN}"} if HF_TOKEN else {}

CASES = [   # OpenFake core/test 里的行号、它展示的内容，以及我们为条件 2 手工打的标签
    {"row": 6831,  "what": "Real Madrid press conference", "real_entity": True,  "fiction": False},
    {"row": 29127, "what": "The Daily Show studio",        "real_entity": True,  "fiction": False},
    {"row": 3251,  "what": "anonymous man, Bernie shirt",  "real_entity": False, "fiction": False},
    {"row": 6814,  "what": "anime girl on a beach",        "real_entity": False, "fiction": True},
    {"row": 6811,  "what": "bearded dragon",               "real_entity": False, "fiction": False},
    {"row": 6851,  "what": "Mini Cooper under a tarp",     "real_entity": False, "fiction": False},
    {"row": 6804,  "what": "parking-garage mirror",        "real_entity": False, "fiction": False},
]

def fetch_example(ex):
    """图像字节加上数据集的标签和生成器；如果我们已经有，就从磁盘读。"""
    path = os.path.join(SAMPLE_DIR, f"{SPLIT}-{ex['row']}")
    if not os.path.exists(path + ".json"):
        q = urllib.parse.urlencode({"dataset": DATASET_ID, "config": SUBSET, "split": SPLIT,
                                    "offset": ex["row"], "length": 1})
        row = json.loads(http_get(f"{VIEWER}/rows?{q}", HF_HEADERS))["rows"][0]["row"]
        os.makedirs(SAMPLE_DIR, exist_ok=True)
        with open(path + ".img", "wb") as f:
            f.write(http_get(row["image"]["src"]))   # 指向这一张图像的签名链接
        with open(path + ".json", "w") as f:
            json.dump({"label": str(row["label"]).lower(), "generator": row.get("model")}, f)
    ex.update(json.load(open(path + ".json")))
    ex["raw"] = open(path + ".img", "rb").read()

for c in CASES:
    fetch_example(c)
    c["gold"] = c["label"] == "fake" and c["real_entity"] and not c["fiction"]

show_table([{"image": thumb(c["raw"]), "case": c["what"], "row": c["row"],
             "dataset label": c["label"], "generator": c["generator"],     # 来自 OpenFake
             "real entity": c["real_entity"], "fiction": c["fiction"],     # 我们的，手工标注
             "gold": "VIOLATION" if c["gold"] else "-"} for c in CASES])

## 3. 方法

三个工具，各自回答它适合回答的条件：

| 工具 | 它是什么 | 回答 | 成本 |
|---|---|---|---|
| **A — detector** | 一个训练来区分真实照片和 AI 图像的图像分类器；输出 `p_fake` | 条件 1：它是 AI 生成的吗？ | CPU 上约 0.2 秒，免费 |
| **B — VLM** | 一个视觉语言模型，被要求看图像并填一张 JSON 表单 | 条件 1 和 2，外加一个*可见的* AI 标签 | 每张图像一次 API 调用 |
| **D — provenance** | 读取文件字节，找机器可读的「由 AI 制作」标记 | 条件 3：它是否在文件里被披露过？ | 微秒级 |

**一句话说清 D：** 它根本不看画面，只看文件的元数据。像 ChatGPT 和 Gemini 这样的生成器会在输出文件里打上 IPTC 术语 `trainedAlgorithmicMedia`（「由训练过的模型制作」）和一个签名的 C2PA manifest（一个「content credential」）。如果标记在，文件已经自我披露为 AI，所以它不可能是*未披露*的 deepfake。D 是最便宜的工具，也是最脆弱的：一次截图或另存就会把元数据丢掉。

从这三个工具我们搭出三个系统，每个都是对「要不要标记出来供复核？」的不同回答：

| 系统 | 在什么情况下标记 |
|---|---|
| **A only** | detector 说是 fake（最朴素的构建：每个 fake 都是「违规」） |
| **B end to end** | VLM 说是 fake、真实实体、非虚构、无可见标签（一个模型做完它能看到的全部判断） |
| **C hybrid** | detector 说是 fake **且** VLM 说是真实实体、非虚构、无可见标签 **且**无元数据标记 |

B 和 C 用的是同一个 VLM 回答；它们的区别只在于*谁来决定「它是不是 fake」*：VLM 自己，还是 detector。

## 4. 三个工具，逐个来看

每个工具都立刻加载并在那七张图像上跑一遍，这样在任何组合之前，你就能看到它说了什么。

### D —— provenance 检查

我们寻找 IPTC 术语和一个内嵌的 C2PA box。找到 C2PA box 不等于*验证*它：那需要把签名对照一份信任列表来检查（练习 5）。像 SynthID 这样的不可见水印在这里根本没法检查。

In [ ]:
def provenance(raw):
    return {"ai_metadata": b"trainedAlgorithmicMedia" in raw,
            "c2pa_manifest": b"jumb" in raw and b"c2pa" in raw}

for c in CASES:
    c["prov"] = provenance(c["raw"])
print(f"AI metadata in {sum(c['prov']['ai_metadata'] for c in CASES)}/{len(CASES)} files, "
      f"C2PA manifest in {sum(c['prov']['c2pa_manifest'] for c in CASES)}/{len(CASES)}")

七张里没有一张带标记：OpenFake 存的是朴素图像，也就是大多数被重新上传的内容在平台上看起来的样子，这也是我们能假设「无披露」的原因。D 的价值在第 6 步体现，那时会用一个直接从生成器下载的文件。

### A —— 开箱即用的 detector

你不需要自己训练一个 detector 才能用它。两个来自不同代际的预训练分类器，都按原样使用，阈值取 0.5：

- **`A_openfake`**：[OpenFake](https://huggingface.co/datasets/ComplexDataLab/OpenFake) 作者随数据集发布的 SwinV2 detector，在 [`bendavidsteel/openfake-swinv2-base`](https://huggingface.co/bendavidsteel/openfake-swinv2-base)。它在 OpenFake 的 train split 上训练过，所以它认识这些生成器；可以预期它在这里的表现会好于在你的平台上的表现。
- **`A_cnn`**：**CNNDetection**（[Wang et al., CVPR 2020](https://github.com/peterwang512/CNNDetection)，*「CNN-generated images are surprisingly easy to spot… for now」*），后来的论文把它列为「Blur+JPEG (0.5)」的那个经典基线。一个在 ProGAN fake 上训练的普通 ResNet-50，用随机模糊和 JPEG 压缩作为数据增强（施加于一半训练图像，所以是 0.5）。它从没见过扩散模型，而 OpenFake 几乎全是扩散模型。论文标题里已经写了「for now」。

| | `A_openfake` | `A_cnn` |
|---|---|---|
| 架构 | SwinV2-base（一个 transformer） | ResNet-50（一个 CNN） |
| 训练于 | OpenFake：2025–2026 年的扩散模型 | ProGAN（2018 年的 GAN）图像 |
| 权重 | Hugging Face，348 MB | 作者的 Dropbox，282 MB |

In [ ]:
import torch, torchvision
from transformers import AutoImageProcessor, AutoModelForImageClassification

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
THRESHOLD = 0.5

class Detector:
    def __init__(self, repo, fake_label, temperature=1.0):
        self.proc = AutoImageProcessor.from_pretrained(repo)
        self.model = AutoModelForImageClassification.from_pretrained(repo).to(DEVICE).eval()
        self.fake = {v.lower(): int(k) for k, v in self.model.config.id2label.items()}[fake_label]
        self.temperature = temperature

    @torch.no_grad()
    def __call__(self, raws):
        x = self.proc(images=[view(r) for r in raws], return_tensors="pt").to(DEVICE)
        p = (self.model(**x).logits.float() / self.temperature).softmax(-1)[:, self.fake]
        return p.cpu().tolist()

class CNNDetection:
    """Wang et al. 2020：一个 torchvision ResNet-50，它唯一的输出就是 'fake' 的 logit。"""
    URL = "https://www.dropbox.com/s/2g2jagq2jn1fd0i/blur_jpg_prob0.5.pth?dl=1"   # 来自该仓库的下载脚本
    PATH = "cnndetection_blur_jpg_prob0.5.pth"

    def __init__(self):
        if not os.path.exists(self.PATH):
            with open(self.PATH, "wb") as f:
                f.write(http_get(self.URL))
        self.model = torchvision.models.resnet50(num_classes=1)
        weights = torch.load(self.PATH, map_location="cpu", weights_only=True)   # 只要 tensor，不要 pickle 进来的代码
        self.model.load_state_dict(weights["model"])
        self.model.to(DEVICE).eval()
        self.norm = torchvision.transforms.Compose([torchvision.transforms.ToTensor(),
            torchvision.transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

    @torch.no_grad()
    def __call__(self, raws):
        # 一次一张、最大 1024 px：它找的是细粒度的像素痕迹，所以能看到比其他工具更大的画面
        return [self.model(self.norm(view(r, 1024))[None].to(DEVICE)).sigmoid().item() for r in raws]

DETECTORS = {
    "A_openfake": Detector("bendavidsteel/openfake-swinv2-base", "fake", temperature=2.0),  # 它的 model card：把 logits 除以 2
    "A_cnn": CNNDetection(),
}
DET = "A_openfake"                 # A 和 C 系统使用的那个 detector

for name, d in DETECTORS.items():
    for c, p in zip(CASES, d([c["raw"] for c in CASES])):
        c.setdefault("det", {})[name] = p

show_table([{"image": thumb(c["raw"]), "case": c["what"], "dataset label": c["label"],
             **{f"{n} p_fake": f"{c['det'][n]:.2f} {mark((c['det'][n] >= THRESHOLD) == (c['label'] == 'fake'))}"
                for n in DETECTORS}} for c in CASES])

✓ 表示 detector 在 0.5 阈值下与数据集的 `real`/`fake` 标签一致。可以预期 `A_cnn` 几乎会把所有东西都判成 real，包括 fake：在我们本地的运行里，七张全部低于 0.01。这不是 bug，而是那堂课：detector 只认得它训练过的那些生成器，而一个 2020 年的 GAN detector 从没见过扩散模型。架构远没有训练数据重要。detector 也只回答条件 1：那张动漫插画应该和 deepfake 一样被判定为 fake。

> **⚠️ 生产现实 ——** OpenFake detector 的 model card 报告说，在社交媒体上它会把很多**真实的**文字图形（截图、新闻下方字幕条、meme）判成 AI：在他们的测试里 35 个中有 22 个。如果你的平台满是截图，那就是你的误报率，而 OpenFake 上的任何数字都显示不出来。

### B —— 一个 VLM

每张图像一次调用，要求它回答 VLM 能看到的每一个条件。它读不了文件元数据，这很符合现实：VLM 看的是像素。`evidence` 那句话是给人工复核者看的，**永远不参与打分**：一个听起来合理的理由不等于是正确的理由。回答按图像的哈希缓存，所以重跑不花任何成本。

In [ ]:
import groq

VLM_PROMPT = (
    "You audit images under EU AI Act Article 50(4). Look at the image and reply with ONLY a JSON object:\n"
    '{"p_fake": <0-1, probability the image is AI-generated or AI-manipulated; ordinary compression, '
    "resizing or filters are NOT manipulation>,\n"
    ' "real_entity": <true if it depicts an existing, identifiable person, place, object or event '
    "and could falsely appear authentic>,\n"
    ' "evident_fiction": <true if it is evidently artistic, satirical, fantastical or fictional>,\n'
    ' "visible_ai_label": <true if the image itself shows a visible notice that it is AI-generated>,\n'
    ' "evidence": "<one short sentence>"}'
)

def parse_json(text):
    m = re.search(r"\{.*\}", text or "", re.DOTALL)
    try:
        return json.loads(m.group()) if m else None
    except json.JSONDecodeError:
        return None

def vlm_call(raw):
    buf = io.BytesIO()
    view(raw).save(buf, format="PNG")
    resp = client.chat.completions.create(
        model=VISION_MODEL, max_tokens=1024, temperature=0,   # 给会先推理的模型留出余量
        messages=[{"role": "user", "content": [
            {"type": "text", "text": VLM_PROMPT},
            {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()}}]}])
    out = parse_json(resp.choices[0].message.content) or {}
    try:
        p = min(max(float(out["p_fake"]), 0.0), 1.0)
    except (KeyError, TypeError, ValueError):
        p = None                                  # 无法解析：如实显示，不去猜
    return {"p_fake": p,
            **{k: out.get(k) is True for k in ("real_entity", "evident_fiction", "visible_ai_label")},
            "evidence": str(out.get("evidence", ""))[:200]}

def vlm_judge(raw):
    return cached(f"vlm|{VISION_MODEL}|{hashlib.sha256(raw).hexdigest()[:16]}", lambda: vlm_call(raw))

for c in CASES:
    try:
        c["vlm"] = vlm_judge(c["raw"])
    except groq.RateLimitError as e:          # 客户端已经重试过了；通常是每日上限
        print(f"⏸ VLM rate limit: {str(e)[:150]}\n  Rerun this cell later; finished calls are cached.")
        break

def vlm_p(v):
    return "n/a" if v["p_fake"] is None else f"{v['p_fake']:.2f}"

show_table([{"image": thumb(c["raw"]), "case": c["what"],
             "p_fake": f"{vlm_p(c['vlm'])} {mark(((c['vlm']['p_fake'] or 0) >= 0.5) == (c['label'] == 'fake'))}",
             "real entity": f"{c['vlm']['real_entity']} {mark(c['vlm']['real_entity'] == c['real_entity'])}",
             "fiction": f"{c['vlm']['evident_fiction']} {mark(c['vlm']['evident_fiction'] == c['fiction'])}",
             "evidence": c["vlm"]["evidence"]} for c in CASES if "vlm" in c])

✓ 表示与数据集标签（`p_fake`）以及与我们的手工标签（真实实体、虚构）一致。它认出真实实体了吗？它看出那张动漫是虚构了吗？这些就是它在这里要做的阅读理解任务。它的 `p_fake` 是它写出来的一个 token，不是校准过的概率：可以预期在真正真实的照片上看到整数般的数值和很有把握的 evidence。

## 5. 对比这三个系统

现在把三个工具的输出组合成第 3 步里的三个系统，并各自对着 gold 打分。每张图像一行：来自各模型的原始分数，然后每个系统的判定。✓ 表示系统与 gold 一致，✗ 表示不一致。

In [ ]:
SYSTEMS = ["A only", "B end to end", "C hybrid"]

def flags(det, vlm, prov):
    is_fake = det[DET] >= THRESHOLD
    meaning = vlm["real_entity"] and not vlm["evident_fiction"] and not vlm["visible_ai_label"]
    return {"A only": is_fake,
            "B end to end": (vlm["p_fake"] or 0) >= 0.5 and meaning,
            "C hybrid": is_fake and meaning and not prov["ai_metadata"]}

done = [c for c in CASES if "vlm" in c]       # 全部七张，除非 VLM 撞上了它的上限
for c in done:
    c["flags"] = flags(c["det"], c["vlm"], c["prov"])

rows = [{"image": thumb(c["raw"]), "case": c["what"], "gold": "VIOLATION" if c["gold"] else "-",
         **{n: f"{c['det'][n]:.2f}" for n in DETECTORS}, "VLM p_fake": vlm_p(c["vlm"]),
         **{s: f"{'FLAG' if c['flags'][s] else '-'} {mark(c['flags'][s] == c['gold'])}" for s in SYSTEMS}}
        for c in done]
if rows:
    rows.append({k: "" for k in rows[0]} | {"case": "cases right",
                 **{s: f"{sum(c['flags'][s] == c['gold'] for c in done)}/{len(done)}" for s in SYSTEMS}})
show_table(rows)

一列一列地读：

- **A only** 会把每个 fake 都标记出来，所以它也会「抓到」那个匿名男子和那张动漫图像：每一个都是复核者必须清掉的误报。
- **B 和 C** 能清掉那两个，前提是 VLM 读对了：这就是 detector 做不了的*含义*检查。B 和 C 意见不同的地方，就是「它是不是 fake」这个判断：VLM 的对照 detector 的。
- 那个**穿「Bernie for President」T 恤的匿名男子**故意是一个需要判断的案例：一位真实政客被*点名*了，但没有任何真人被*描绘*。我们把它标为非 deepfake；如果你的复核者会不同意，那这是一个政策性决定，要在度量任何东西之前就写下来。

七个案例是演示，不是度量。要选定一个设计，你会对每个条件都评估几百个案例（练习 1）。

## 6. 试试你自己的图像

给一个图像 URL，或一个本地路径（例如通过 Colab 的文件面板上传之后）。这个单元格会跑每个模型，用一张表展示每种方法对这张图像的说法，然后给出推荐系统 C hybrid 的**判定**及其理由。

试试你手机里的一张照片、一张直接从 ChatGPT 或 Gemini 下载的图像（它们的文件带 C2PA 和 IPTC 标记，D 应该能找到），然后再试试同一张图像的截图（标记没有了）。

> **⚠️ 隐私 ——** VLM 这一步会把图像发到 Groq 的 API。不要使用私人的照片，也不要上传任何你不愿意公开的东西。

In [ ]:
IMAGE = "https://upload.wikimedia.org/wikipedia/commons/thumb/a/a8/Tour_Eiffel_Wikimedia_Commons.jpg/960px-Tour_Eiffel_Wikimedia_Commons.jpg"

def verdict(det, vlm, prov):
    """C hybrid 的判定，用文字表述：第一个能让这张图像过关的条件，或者一个标记。"""
    p = det[DET]
    if p < THRESHOLD:
        return False, f"the detector says it's a real photo (p_fake {p:.2f})"
    if prov["ai_metadata"]:
        return False, "AI-generated, but the file discloses it in its metadata"
    if vlm["visible_ai_label"]:
        return False, "AI-generated, but it carries a visible AI label"
    if vlm["evident_fiction"]:
        return False, "AI-generated, but evidently fiction or art"
    if not vlm["real_entity"]:
        return False, "AI-generated, but it doesn't depict a real, identifiable person or event"
    return True, "AI-generated, depicts a real entity, and isn't disclosed"

raw = http_get(IMAGE) if IMAGE.startswith("http") else open(IMAGE, "rb").read()
display(view(raw, 320))
det = {name: d([raw])[0] for name, d in DETECTORS.items()}
prov = provenance(raw)
vlm = vlm_judge(raw)
f = flags(det, vlm, prov)

ai = lambda p: "AI-generated" if p >= THRESHOLD else "real"
show_table(
    [{"method": n, "kind": "model", "what it says": f"p_fake {p:.2f}: {ai(p)}"} for n, p in det.items()]
    + [{"method": "VLM", "kind": "model",
        "what it says": f"p_fake {vlm_p(vlm)}: {ai(vlm['p_fake'] or 0)}; real entity {vlm['real_entity']}, "
                        f"fiction {vlm['evident_fiction']}, visible AI label {vlm['visible_ai_label']}. {vlm['evidence']}"},
       {"method": "D provenance", "kind": "file check",
        "what it says": f"AI metadata {prov['ai_metadata']}, C2PA manifest {prov['c2pa_manifest']}"}]
    + [{"method": s, "kind": "system", "what it says": "FLAG for review" if f[s] else "no flag"} for s in SYSTEMS])

flag, why = verdict(det, vlm, prov)
print(f"VERDICT (C hybrid): {'⚠️ FLAG for human review' if flag else '✅ no flag'}, because {why}.")
if (det[DET] >= THRESHOLD) != ((vlm["p_fake"] or 0) >= 0.5):
    print("Note: the detector and the VLM disagree on whether it's AI-generated.")

## 这应该告诉你什么

拿这些去对照你自己的输出，而不是凭信接受：

- **检测是 detector 的活儿，但仅限于它见过的生成器。** OpenFake detector 在这里看起来非常出色，因为它就在这个数据集上训练过；2020 年的那个 CNN 展示了在比它训练数据更新的生成器上会发生什么。你的生产 detector 永远在两者之间，并且正向第二种漂移。
- **合规不等于检测。**「A only」会把每个 fake 都标记出来，不管它是真实实体还是动漫。只有一个同时检查含义的系统才能放行那张动漫，也只有 provenance 能放行一个自我披露过的文件（练习 7）。
- **元数据很脆弱。** 任何重新编码都会把它抹掉（试试第 6 步的截图）。可见标签和鲁棒的水印能存活；只靠元数据不是一种披露策略。
- **VLM 的价值在含义上。** 真实实体、讽刺、可见标签：这些都是阅读理解任务，像素分类器在那里无话可说。

> **⭐ 关键收获 ——** 把法律问题拆解成构成它的那些条件，给每个条件配一个最能回答它的最便宜工具，并对每一块和整体都做度量。在这里，那就是用 detector 回答*它是不是 fake*，用 provenance 回答*它是否披露过*，用 VLM 回答*它描绘了什么*，用人工回答*这是否违规*。让一个模型回答整个问题，建起来更简单，但信任起来更难。

> **🔵 面试信号 ——** 混合方案还有一根 B 没有的成本杠杆：对一切都跑那个便宜的 detector，**只**对它标记出的图像调用 VLM。如果 detector 标记了 5% 的流量，VLM 花费会降约 20 倍。在设计评审里把这个数字说出来。

## 练习

1. **从演示到度量。** 为每个条件建一个 50+ 案例的黄金集：来自两个不同来源的真实照片、来自比 detector 训练数据更新的生成器的 fake、真实实体 fake（搜 OpenFake 里 `prompt_kind == "published_prompt"` 的行：用户写的 prompt 会保留自动字幕丢弃的那些名字）。报出每个系统的 precision 和 recall（[04/01](../04-evals/01-golden-sets.zh.ipynb)）。七个案例得出的排名还站得住吗？
2. **校准阈值。** 0.5 是这些 detector 的默认值，不是一个决定。从*你的*平台那类来源取 100 张真实照片打分，为每个 detector 设定阈值，使其中至多 5% 被标记。它离 0.5 有多远？
3. **你自己的照片集。** 把你自己的 20 张图像放进一个文件夹，配一个 CSV（`file,label,real_entity,fiction`），用第 6 步的代码跑一遍。要包含手机照片、截图，以及直接从生成器下载的 AI 图像。哪个系统错的次数最多，错在什么上？
4. **训练你自己的 detector。** 冻结一个 CLIP 图像编码器，给 200 张 OpenFake 训练图像做 embedding，在 embedding 上拟合一个逻辑回归（一个线性探针，UnivFD 的配方）。在你的黄金集上，以及两者都没见过的生成器上，它离 `A_openfake` 有多近？
5. **真正的 C2PA。** 安装 `c2pa-python` 包，把 `provenance` 里的字节检查换成真正的 manifest 读取和签名验证。用一张直接从生成器下载的、带 C2PA 签名的图像测试它。「manifest 存在但签名无效」对合规判定意味着什么？
6. **混合方案的经济账。** 改动第 6 步，让 VLM 只在 detector 标记时才被调用。在每月 500 万次上传、其中 1% 是未披露的 deepfake 的情况下，B 与加闸门的混合方案各自的 VLM 花费是多少，复核者的工作量是多少（标记数 × 1 分钟）？写出那一句设计文档建议。
7. **已披露的副本。** 条件 3 说的是*已发布的文件*。把一张 deepfake 另存一份，在上面画一个可见的「AI-generated」框（`ImageDraw`），再把另一张存成 PNG，让它 XMP 元数据里带 IPTC 的 `trainedAlgorithmicMedia` 术语（`PngInfo.add_itxt("XML:com.adobe.xmp", ...)`）。每个系统还会标记哪一份副本？哪种披露能扛过一次截图？